# Matched latent representation

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/lss').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from lss.plotting import apply_editorial_style, PAPER_COLORS
apply_editorial_style()
RESULTS = ROOT / 'notebooks/results/latent_matched_study'
frames, status = [], []
for recipe_path in sorted(RESULTS.glob('*/recipe.json')):
    recipe = json.loads(recipe_path.read_text())
    run = recipe_path.parent
    source = recipe['source']['source_name']
    dim = recipe['config']['ae_config']['latent_dim']
    seed = recipe['config']['model_seed']
    complete = (run / 'completed.json').is_file()
    status.append({'model': source, 'dimension': dim, 'seed': seed, 'completed': complete})
    if complete:
        frame = pd.read_csv(run / 'validation_summary.csv')
        frame['model'], frame['dimension'], frame['seed'] = source, dim, seed
        frames.append(frame)
scores = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
display(pd.DataFrame(status).groupby(['model', 'dimension']).completed.agg(['sum', 'count']) if status else pd.DataFrame())


In [ ]:
if not scores.empty:
    endpoint = scores.loc[(scores.frame == 100) & (scores.control == 'ae')].copy()
    display(endpoint[['source', 'model', 'dimension', 'seed', 'position_mse',
                      'strain_x_mae', 'strain_y_mae', 'p_ratio_r2', 'p_ratio_mae',
                      'valid', 'true_valid', 'total', 'target_variance']].sort_values(['source', 'model', 'dimension', 'seed']))


In [ ]:
if not scores.empty:
    sources = ['reid', 'depablo_low_temp', 'depablo_mixed_temp', 'lj_noisy']
    labels = ['Reid', 'dePablo low-T', 'dePablo mixed-T', 'noisy-LJ']
    styles = [('shared3', 'Shared 3', PAPER_COLORS['blue']),
              ('shared4', 'Shared 4', PAPER_COLORS['purple']),
              ('individual', 'Individual', PAPER_COLORS['green'])]
    for metric, ylabel in [('p_ratio_r2', 'Validation p-ratio R²'),
                           ('position_mse', 'Validation displacement MSE')]:
        fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), squeeze=False)
        for ax, source, label in zip(axes.flat, sources, labels):
            subset = scores.loc[(scores.source == source) & (scores.frame == 100) & (scores.control == 'ae')]
            for model, model_label, color in styles:
                selected = subset.loc[subset.model == (source if model == 'individual' else model)]
                for _, seed_scores in selected.groupby('seed'):
                    seed_scores = seed_scores.sort_values('dimension')
                    ax.plot(seed_scores.dimension, seed_scores[metric], color=color, alpha=.25, linewidth=1)
                means = selected.groupby('dimension')[metric].mean()
                if len(means):
                    ax.plot(means.index, means.values, 'o-', color=color, label=model_label)
            if metric == 'p_ratio_r2':
                ax.axhline(0, color=PAPER_COLORS['slate'], linewidth=.7, linestyle=':')
            else:
                ax.set_yscale('log')
            ax.set(title=label, xlabel='Latent dimension', xticks=[2, 4, 6, 8])
        axes[0, 0].set_ylabel(ylabel)
        axes[0, 0].legend()
        fig.tight_layout()
        plt.show()


In [ ]:
if not scores.empty:
    display(scores.loc[(scores.frame == 100) & (scores.model == 'shared4'),
                       ['source', 'dimension', 'seed', 'control', 'position_mse', 'p_ratio_r2', 'valid', 'total']]
            .sort_values(['source', 'dimension', 'seed', 'control']))
